# 08 · Uncertainty and resolution — worked solutions

These solutions are most useful after attempting the chapter exercises. Each
section states the reasoning before the calculation; numerical values depend on
the compact, fixed-seed setup below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=30.0,
    length=24_000.0, width=12_000.0, n_length=4, n_width=3,
)
i = np.arange(fault.n_patches) % 4
j = np.arange(fault.n_patches) // 4
true_slip = np.exp(-((i - 1.5) / 1.2) ** 2 - ((j - 1.0) / 0.9) ** 2)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 6), np.linspace(33.90, 34.12, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.004
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="solution_gnss",
)


## 1. Checker scale and strength

Fine checkerboards project into weak modes and recover poorly. Stronger
regularization improves stability but increases spatial averaging.

## 2. Spike depth

Deep spikes have broad, similar surface responses, so their resolution kernels
spread over more patches.

## 3. Moment uncertainty

Moment is a linear weighted sum, so propagate the full covariance including
off-diagonal terms.

## 4. The identity limit

Only a full-rank, noise-free small system approaches identity as lambda tends
to zero; real distributed problems need not.


In [ ]:
result = geodef.solve(
    fault, datasets=gnss, components="dip", regularization="laplacian",
    regularization_strength=1.0,
)
Cm = geodef.invert.model_covariance(result, fault, gnss)
R = geodef.invert.model_resolution(result, fault, gnss)
area = fault.areas
moment_sigma = fault.medium.mu * np.sqrt(area @ Cm @ area)
row = R[fault.patch_index(strike_idx=2, dip_idx=1)]
print("moment sigma (N m):", moment_sigma)
print("resolution diagonal:", np.diag(R))
print("selected averaging kernel:", row)


## Interpretation and alternatives

Using the full covariance can increase or decrease a derived uncertainty relative to diagonal-only propagation because correlations have signs.
